# ZynForge Zenith：30 Epoch JouleWeave 对标训练与严格验收

这份 Notebook 的目标是在 **20,000 个 MatPES PBE 2025.2 样本、30 个 epoch**
内尽可能达到旧 JouleWeave RTX5080 训练的能量与力精度。这里的“30”指完整遍历
训练集 30 次，不是只做 30 个 mini-batch；后者没有足够训练量。

相对于资源优先的 compact 配置，本 Notebook 只做以下低物理风险调整：

- 使用三层、80 通道、两个平滑专家和秩 20 的中等容量；
- 物理 batch 2、梯度累积 8，有效 batch 固定为 16，与 JouleWeave 相同；
- 30 轮内从第一轮启用完整能量、力、电荷和磁矩目标，不消耗六轮课程预算；
- 使用与 JouleWeave 相同的 Huber `delta=0.02`、五轮 warmup 和参考能拟合；
- 明确要求保留缺失电子标签的能量/力样本，并用数据指纹验证训练集合；
- 使用统一评估器同时输出 force component-micro 与 structure-macro 指标；
- 同时评价 EMA 与 raw 权重，只按验证集上的预先固定无量纲分数选择部署权重；
- 训练后重新检查 O(3)、反射、平移、排列、保守力、Hessian 和应力导数。

保持不变的物理结构包括：相对坐标、完整半径图、O(3) 分级不可约表示、赝张量、
CG/ACE 高体阶耦合、可正可负路径系数、单一标量势能、自动微分力、C4 截止和短程
ZBL。没有增加独立能量通路或独立 force head。

注意：Notebook 将“达到 JouleWeave”作为可计算的验收条件，而不是预先保证结果。
若未通过，它会保存差距和全部诊断，不会把目标当作已经实现的事实。


## 0. 用户配置

通常只需修改 `ZYNNOVA_PROJECT_ROOT`。默认针对约 16 GB 显存的单卡：
`batch=2, accumulation=8`。若训练探针显存不足，可改为 `batch=1,
accumulation=16`；有效 batch 必须继续保持 16。

为保证与旧 JouleWeave 公平比较，默认启用数据指纹门禁。若数据源的 `main`
分支以后发生变化，Notebook 会停止并报告实际指纹；不要为了得到一条曲线而静默
关闭门禁。只有在明确进行“新数据实验”时才应设置
`ZYNFORGE_REQUIRE_REFERENCE_FINGERPRINT=0`。


In [1]:
import os
from pathlib import Path

os.environ.setdefault("ZYNNOVA_PROJECT_ROOT", "/home/zephyrain/lw/ZynNova")
os.environ.setdefault("ZYNFORGE_HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("ZYNFORGE_HF_HOME", "/home/zephyrain/lw/huggingface")

os.environ.setdefault("ZYNFORGE_RUN_MODE", "train")
os.environ.setdefault("ZYNFORGE_RESUME_CHECKPOINT", "")
os.environ.setdefault("ZYNFORGE_EVALUATE_CHECKPOINT", "")
os.environ.setdefault("ZYNFORGE_EPOCHS", "30")
os.environ.setdefault("ZYNFORGE_DATA_LIMIT", "20000")

os.environ.setdefault("ZYNFORGE_CHARGE_SCHEME", "ddec6")
os.environ.setdefault("ZYNFORGE_USE_CHARGES", "1")
os.environ.setdefault("ZYNFORGE_USE_MAGMOMS", "1")
os.environ.setdefault("ZYNFORGE_LOCAL_JSONL", "")
os.environ.setdefault("ZYNFORGE_OXIDATION_SIDECAR", "")
os.environ.setdefault("ZYNFORGE_OXIDATION_METHOD", "")

# 与 JouleWeave 的 effective batch=16 严格对齐。
os.environ.setdefault("ZYNFORGE_BATCH_SIZE", "8")
os.environ.setdefault("ZYNFORGE_GRAD_ACCUM", "2")
os.environ.setdefault("ZYNFORGE_NUM_WORKERS", "2")
os.environ.setdefault("ZYNFORGE_PROGRESS_INTERVAL", "10")
os.environ.setdefault("ZYNFORGE_MAX_ATOMS", "56")

# 中等容量：只增加合法 irrep 通道、层数和低秩系数自由度。
os.environ.setdefault("ZYNFORGE_HIDDEN_DIM", "64")
os.environ.setdefault("ZYNFORGE_NUM_LAYERS", "3")
os.environ.setdefault("ZYNFORGE_NUM_RADIAL", "16")
os.environ.setdefault("ZYNFORGE_NUM_HEADS", "4")
os.environ.setdefault("ZYNFORGE_NUM_EXPERTS", "2")
os.environ.setdefault("ZYNFORGE_TENSOR_RANK", "16")
os.environ.setdefault("ZYNFORGE_EDGE_RANK", "16")
os.environ.setdefault("ZYNFORGE_CUTOFF_A", "5.0")

os.environ.setdefault("ZYNFORGE_RUN_SOURCE_REGRESSION", "1")
os.environ.setdefault("ZYNFORGE_RUN_RUNTIME_PROBE", "1")
os.environ.setdefault("ZYNFORGE_RUN_FULL_TEST", "1")
os.environ.setdefault("ZYNFORGE_RUN_PHYSICS_CHECK", "1")
os.environ.setdefault("ZYNFORGE_RUN_SECOND_ORDER_CHECK", "1")
os.environ.setdefault("ZYNFORGE_RUN_ASE_CHECK", "1")
os.environ.setdefault("ZYNFORGE_COMPILE_CONSERVATIVE", "0")

# 数据一致性与最终验收。
os.environ.setdefault("ZYNFORGE_REQUIRE_REFERENCE_FINGERPRINT", "1")
os.environ.setdefault(
    "ZYNFORGE_EXPECTED_DATASET_SHA256",
    "e04aa34106cee73aac2d136c5a0a7e8b50f22e079c9968d421b9b05a80d63f13",
)
os.environ.setdefault("ZYNFORGE_REQUIRE_PARITY", "0")

# 尚未由多种子消融证明稳定净收益的可选模块继续关闭。
os.environ.setdefault("ZYNFORGE_USE_SCALE_CONTEXT", "0")
os.environ.setdefault("ZYNFORGE_USE_GATED_FFN", "0")
os.environ.setdefault("ZYNFORGE_USE_PATH_RADIAL", "0")
os.environ.setdefault("ZYNFORGE_USE_ADAPTIVE_RANK", "0")
os.environ.setdefault("ZYNFORGE_USE_PERIODIC_PRIOR", "0")

os.environ.setdefault("ZYNFORGE_USE_QEQ", "0")
os.environ.setdefault("ZYNFORGE_USE_DISPERSION", "0")
os.environ.setdefault("ZYNFORGE_USE_LATENT_EWALD", "0")
os.environ.setdefault("ZYNFORGE_SEED", "42")
os.environ.setdefault("ZYNFORGE_DETERMINISTIC", "0")


'0'

## 1. 环境、接口与硬件检查


In [2]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import random
import subprocess
import sys
import time
from dataclasses import asdict
from datetime import datetime
from pathlib import Path

os.environ.setdefault("HF_ENDPOINT", os.environ["ZYNFORGE_HF_ENDPOINT"])
os.environ.setdefault("HF_HOME", os.environ["ZYNFORGE_HF_HOME"])
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_MODULE_LOADING", "LAZY")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".matplotlib-cache").resolve()))

PROJECT_ROOT = Path(os.environ["ZYNNOVA_PROJECT_ROOT"]).expanduser().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(f"不是有效的 ZynNova 项目根目录: {PROJECT_ROOT}")
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

RUN_MODE = os.environ["ZYNFORGE_RUN_MODE"].strip().lower()
if RUN_MODE not in {"train", "resume", "evaluate"}:
    raise ValueError("ZYNFORGE_RUN_MODE 必须是 train / resume / evaluate")

EPOCHS = int(os.environ["ZYNFORGE_EPOCHS"])
DATA_LIMIT = int(os.environ["ZYNFORGE_DATA_LIMIT"])
BATCH_SIZE = int(os.environ["ZYNFORGE_BATCH_SIZE"])
GRAD_ACCUM = int(os.environ["ZYNFORGE_GRAD_ACCUM"])
NUM_WORKERS = int(os.environ["ZYNFORGE_NUM_WORKERS"])
MAX_ATOMS = int(os.environ["ZYNFORGE_MAX_ATOMS"])
PROGRESS_INTERVAL = int(os.environ["ZYNFORGE_PROGRESS_INTERVAL"])
SEED = int(os.environ["ZYNFORGE_SEED"])

HIDDEN_DIM = int(os.environ["ZYNFORGE_HIDDEN_DIM"])
NUM_LAYERS = int(os.environ["ZYNFORGE_NUM_LAYERS"])
NUM_RADIAL = int(os.environ["ZYNFORGE_NUM_RADIAL"])
NUM_HEADS = int(os.environ["ZYNFORGE_NUM_HEADS"])
NUM_EXPERTS = int(os.environ["ZYNFORGE_NUM_EXPERTS"])
TENSOR_RANK = int(os.environ["ZYNFORGE_TENSOR_RANK"])
EDGE_RANK = int(os.environ["ZYNFORGE_EDGE_RANK"])
CUTOFF_A = float(os.environ["ZYNFORGE_CUTOFF_A"])

RUN_SOURCE_REGRESSION = os.environ["ZYNFORGE_RUN_SOURCE_REGRESSION"] == "1"
RUN_RUNTIME_PROBE = os.environ["ZYNFORGE_RUN_RUNTIME_PROBE"] == "1"
RUN_FULL_TEST = os.environ["ZYNFORGE_RUN_FULL_TEST"] == "1"
RUN_PHYSICS_CHECK = os.environ["ZYNFORGE_RUN_PHYSICS_CHECK"] == "1"
RUN_SECOND_ORDER_CHECK = os.environ["ZYNFORGE_RUN_SECOND_ORDER_CHECK"] == "1"
RUN_ASE_CHECK = os.environ["ZYNFORGE_RUN_ASE_CHECK"] == "1"
COMPILE_CONSERVATIVE = os.environ["ZYNFORGE_COMPILE_CONSERVATIVE"] == "1"
DETERMINISTIC = os.environ["ZYNFORGE_DETERMINISTIC"] == "1"
REQUIRE_REFERENCE_FINGERPRINT = (
    os.environ["ZYNFORGE_REQUIRE_REFERENCE_FINGERPRINT"] == "1"
)
EXPECTED_DATASET_SHA256 = os.environ["ZYNFORGE_EXPECTED_DATASET_SHA256"].strip()
REQUIRE_PARITY = os.environ["ZYNFORGE_REQUIRE_PARITY"] == "1"

USE_CHARGES = os.environ["ZYNFORGE_USE_CHARGES"] == "1"
USE_MAGMOMS = os.environ["ZYNFORGE_USE_MAGMOMS"] == "1"
USE_QEQ = os.environ["ZYNFORGE_USE_QEQ"] == "1"
USE_DISPERSION = os.environ["ZYNFORGE_USE_DISPERSION"] == "1"
USE_LATENT_EWALD = os.environ["ZYNFORGE_USE_LATENT_EWALD"] == "1"
USE_SCALE_CONTEXT = os.environ["ZYNFORGE_USE_SCALE_CONTEXT"] == "1"
USE_GATED_FFN = os.environ["ZYNFORGE_USE_GATED_FFN"] == "1"
USE_PATH_RADIAL = os.environ["ZYNFORGE_USE_PATH_RADIAL"] == "1"
USE_ADAPTIVE_RANK = os.environ["ZYNFORGE_USE_ADAPTIVE_RANK"] == "1"
USE_PERIODIC_PRIOR = os.environ["ZYNFORGE_USE_PERIODIC_PRIOR"] == "1"
CHARGE_SCHEME = os.environ["ZYNFORGE_CHARGE_SCHEME"].strip().lower()

if min(
    EPOCHS, DATA_LIMIT, BATCH_SIZE, GRAD_ACCUM, MAX_ATOMS,
    HIDDEN_DIM, NUM_LAYERS, NUM_RADIAL, NUM_HEADS,
) < 1:
    raise ValueError("训练规模和模型容量参数必须为正")
if EPOCHS != 30:
    print(f"警告：当前 EPOCHS={EPOCHS}；JouleWeave 对标目标定义在 30 epoch。")
if BATCH_SIZE * GRAD_ACCUM != 16:
    raise ValueError(
        "公平对标要求 effective batch=16。推荐 2×8；显存不足时使用 1×16。"
    )
if HIDDEN_DIM % NUM_HEADS:
    raise ValueError("hidden_dim 必须能被 attention heads 整除")
if USE_QEQ and USE_LATENT_EWALD:
    raise ValueError("QEq 与 latent Ewald 不能在无可辨识监督下叠加")
if USE_QEQ:
    raise ValueError(
        "本 Notebook 训练周期晶体，不使用有限体系 QEq 的最小镜像近似。"
    )

run_stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
OUTPUT_ROOT = (PROJECT_ROOT.parent / "ZynForge_Zenith_JouleWeave_Parity").resolve()
WORKSPACE_ROOT = OUTPUT_ROOT / "workspace"
RESULTS_ROOT = OUTPUT_ROOT / "results"
# 独立目录避免读取 require_electronic_labels=True 创建的同名旧缓存。
CACHE_ROOT = OUTPUT_ROOT / "matpes-reference-protocol-cache"
for directory in (OUTPUT_ROOT, WORKSPACE_ROOT, RESULTS_ROOT, CACHE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)


In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

from zynnova.ml.common import move_to_device, resolve_device
from zynnova.ml.workspace import MLWorkspace
from zynnova.ml.zynforge.data import (
    MatPESElectronicConfig,
    load_matpes_electronic_dataset,
    validate_checkpoint_provenance,
    validate_electronic_provenance,
)
from zynnova.ml.zynforge.field import (
    ZynFieldConfig,
    ZynFieldDataConfig,
    ZynFieldModelConfig,
    ZynFieldPotential,
    ZynFieldTrainConfig,
    architecture_efficiency_report,
    check_conservative_forces,
    check_hessian_symmetry,
    check_o3_equivariance,
    check_permutation_translation_invariance,
    check_stress_energy_derivative,
    formal_completeness_certificate,
    load_zynfield,
    prepare_zynfield_datamodule,
    resume_zynfield,
    train_zynfield,
    zynfield_calculator,
)
from zynnova.ml.zynforge.field.data import jouleweave_collate

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = not DETERMINISTIC
    torch.backends.cudnn.allow_tf32 = not DETERMINISTIC
    torch.backends.cudnn.benchmark = not DETERMINISTIC
    torch.use_deterministic_algorithms(DETERMINISTIC, warn_only=True)
    torch.set_float32_matmul_precision("high")

DEVICE = resolve_device("auto")
if DEVICE.type == "cuda":
    gpu_name = torch.cuda.get_device_name(DEVICE)
    gpu_memory_gib = torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3
else:
    gpu_name = "CPU"
    gpu_memory_gib = 0.0
    if RUN_MODE in {"train", "resume"}:
        print("警告：当前没有 CUDA；正式 20K 力训练会很慢。")

print("Torch / device      :", torch.__version__, DEVICE)
print("Accelerator          :", gpu_name, f"({gpu_memory_gib:.2f} GiB)")
print("Architecture         : zynforge-zenith")
print("Epochs / data limit  :", EPOCHS, DATA_LIMIT)
print("Micro/effective batch:", BATCH_SIZE, BATCH_SIZE * GRAD_ACCUM)
print("H/L/radial/experts   :", HIDDEN_DIM, NUM_LAYERS, NUM_RADIAL, NUM_EXPERTS)
print("Ranks / cutoff       :", TENSOR_RANK, EDGE_RANK, CUTOFF_A)
print("Deterministic        :", DETERMINISTIC)
print("Conservative compile :", COMPILE_CONSERVATIVE)

def batch_to_model_inputs(batch, *, device=DEVICE, dtype=torch.float32):
    batch = move_to_device(batch, device)
    structure = dict(batch["structure"])
    for key in ("pos", "cell"):
        if key in structure and torch.is_tensor(structure[key]):
            structure[key] = structure[key].to(dtype=dtype)
    structure["positions"] = structure["pos"]
    structure["_return_auxiliary_fields"] = False
    for name in ("total_charge", "spin", "fidelity"):
        if name in batch.get("conditions", {}):
            structure[name] = batch["conditions"][name]
    return batch, structure


Torch / device      : 2.13.0+cu130 cuda
Accelerator          : NVIDIA GeForce RTX 5080 (15.92 GiB)
Architecture         : zynforge-zenith
Epochs / data limit  : 30 20000
Micro/effective batch: 8 16
H/L/radial/experts   : 64 3 16 2
Ranks / cutoff       : 16 16 5.0
Deterministic        : False
Conservative compile : False


## 2. 源码架构与物理门禁

先用很小的同构模型在 CPU 双精度下验证结构，不依赖训练数据或 checkpoint。
一阶检查默认执行；Hessian 与应力有限差分由 `RUN_SECOND_ORDER_CHECK` 控制。


In [4]:
# probe_config = ZynFieldModelConfig.specialist(
#     hidden_dim=16,
#     num_layers=2,
#     num_radial=8,
#     num_attention_heads=4,
#     num_experts=1,
#     expert_top_k=1,
#     max_ell=3,
#     correlation_order=3,
#     tensor_product_rank=4,
#     directional_edge_rank=4,
#     interaction_cutoff_A=4.0,
#     max_neighbors=None,
#     use_pair_chemical_bias=False,
#     use_hybrid_irrep_norm=False,
#     use_learned_residual_scales=False,
#     use_electronic_depth_context=False,
#     use_invariant_scale_context=False,
#     use_invariant_gated_ffn=False,
#     use_path_resolved_radial=False,
#     use_adaptive_rank_gates=False,
#     use_periodic_table_prior=False,
#     extensive_state_conditioning=True,
#     use_magmoms=False,
#     use_charge_head=False,
#     use_zbl=False,
#     use_dispersion=False,
#     use_qeq=False,
# )
# probe_model = ZynFieldPotential(probe_config).double().eval()
# probe_report = architecture_efficiency_report(probe_model)
# probe_certificate = formal_completeness_certificate(probe_model)

# assert probe_config.architecture_name == "zynforge-zenith"
# assert probe_report.single_graph_ace_spine
# assert probe_report.complete_edge_cg_layers == probe_config.num_layers
# assert probe_report.full_correlation_layers == probe_config.num_layers
# assert probe_report.directional_edge_layers == probe_config.num_layers
# assert probe_report.grace_finite_tree_basis_contained
# assert probe_report.single_energy_path
# assert probe_report.complete_radius_graph
# assert probe_report.hard_neighbor_cap is None
# assert probe_report.merged_graded_rms_norm
# assert probe_report.grade_aware_invariant_edge_routing
# assert probe_certificate.complete_node_product_layers == probe_config.num_layers
# assert probe_certificate.all_spatial_cg_paths
# assert probe_certificate.all_rooted_tree_topologies
# assert probe_certificate.signed_graph_path_coefficients
# assert not probe_certificate.graph_path_positive_floor
# assert not probe_certificate.separate_additive_energy_path
# assert probe_certificate.finite_truncation_feature_complete

# forbidden_modules = {"IndependentTopologyWordMixer", "FactorizedInvariantEdgeKernel"}
# present_modules = {module.__class__.__name__ for module in probe_model.modules()}
# assert not (forbidden_modules & present_modules)

# # Test the new grade-aware route away from its zero-residual identity
# # initialization. Otherwise an equivariance test could pass without
# # exercising the learned grade-dependent coefficients.
# route_generator = torch.Generator().manual_seed(SEED + 811)
# with torch.no_grad():
#     for block in probe_model.backbone.blocks:
#         for projection in (
#             block.edge_attention.output_projection,
#             block.edge_attention.grade_projection,
#         ):
#             projection.weight.copy_(
#                 0.10
#                 * torch.randn(
#                     projection.weight.shape,
#                     generator=route_generator,
#                     dtype=torch.float64,
#                 )
#             )

# probe_inputs = {
#     "z": torch.tensor([1, 8, 6], dtype=torch.long),
#     "pos": torch.tensor(
#         [[0.2, 0.3, 0.4], [1.3, 0.4, 0.5], [0.5, 1.6, 0.7]],
#         dtype=torch.float64,
#     ),
#     "batch": torch.zeros(3, dtype=torch.long),
#     "cell": 6.0 * torch.eye(3, dtype=torch.float64).unsqueeze(0),
#     "pbc": torch.zeros((1, 3), dtype=torch.bool),
# }
# force_report = check_conservative_forces(probe_model, probe_inputs)
# permutation_report = check_permutation_translation_invariance(probe_model, probe_inputs)
# rotation = torch.tensor(
#     [[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0]],
#     dtype=torch.float64,
# )
# rotation_report = check_o3_equivariance(probe_model, probe_inputs, rotation)
# reflection_report = check_o3_equivariance(
#     probe_model,
#     probe_inputs,
#     torch.diag(torch.tensor([-1.0, 1.0, 1.0], dtype=torch.float64)),
# )
# for report in (force_report, permutation_report, rotation_report, reflection_report):
#     assert report.passed, report

# source_physics = {
#     "force": asdict(force_report),
#     "permutation_translation": asdict(permutation_report),
#     "rotation": asdict(rotation_report),
#     "reflection": asdict(reflection_report),
# }
# if RUN_SECOND_ORDER_CHECK:
#     hessian_report = check_hessian_symmetry(probe_model, probe_inputs)
#     periodic_probe = dict(probe_inputs)
#     periodic_probe["pbc"] = torch.ones((1, 3), dtype=torch.bool)
#     stress_report = check_stress_energy_derivative(probe_model, periodic_probe)
#     assert hessian_report.passed, hessian_report
#     assert stress_report.passed, stress_report
#     source_physics["hessian"] = asdict(hessian_report)
#     source_physics["stress"] = asdict(stress_report)

# display(pd.Series(probe_report.to_dict()))
# display(pd.Series(probe_certificate.to_dict()))
# print("Source physical gates: PASSED")
# source_regression = {"enabled": RUN_SOURCE_REGRESSION}
# if RUN_SOURCE_REGRESSION:
#     regression_started = time.perf_counter()
#     subprocess.run(
#         [sys.executable, "-m", "pytest", "-q", str(PROJECT_ROOT / "tests")],
#         cwd=PROJECT_ROOT,
#         check=True,
#     )
#     source_regression.update({
#         "status": "passed",
#         "seconds": time.perf_counter() - regression_started,
#     })
#     print("Full source regression: PASSED")
# del probe_model
# gc.collect()

## 3. 加载与 JouleWeave 完全相同的 MatPES 样本

旧训练保留所有合格的能量/力结构，并通过 mask 使用可用电子标签。因此这里明确设置
`require_electronic_labels=False`。加载后对 ID、划分、结构数组和四类标签计算 SHA-256；
默认必须与已经审计的 JouleWeave 20K 数据指纹完全一致。


In [5]:
local_text = os.environ["ZYNFORGE_LOCAL_JSONL"].strip()
local_jsonl = Path(local_text).expanduser().resolve() if local_text else None
sidecar_text = os.environ["ZYNFORGE_OXIDATION_SIDECAR"].strip()
oxidation_sidecar = Path(sidecar_text).expanduser().resolve() if sidecar_text else None
oxidation_method = os.environ["ZYNFORGE_OXIDATION_METHOD"].strip() or None

matpes_config = MatPESElectronicConfig(
    profile="rtx5080",
    limit=DATA_LIMIT,
    seed=SEED,
    hf_endpoint=os.environ["HF_ENDPOINT"],
    hf_home=Path(os.environ["HF_HOME"]),
    cache_directory=CACHE_ROOT,
    max_atoms=MAX_ATOMS,
    max_neighbors=40,
    use_stress=False,
    use_charges=USE_CHARGES,
    charge_scheme=CHARGE_SCHEME,
    use_magmoms=USE_MAGMOMS,
    oxidation_sidecar=oxidation_sidecar,
    oxidation_label_method=oxidation_method,
    require_electronic_labels=False,
)
dataset = load_matpes_electronic_dataset(matpes_config, local_jsonl=local_jsonl)
all_samples = list(dataset.samples)
provenance = validate_electronic_provenance(all_samples, matpes_config)

if not all_samples:
    raise RuntimeError("没有找到任何通过筛选的 MatPES 样本。")
if len(all_samples) != int(dataset.report.accepted):
    raise RuntimeError("加载报告与实际接受样本数不一致")

def _hash_array(digest, value):
    array = np.asarray(value)
    digest.update(str(array.dtype).encode())
    digest.update(str(tuple(array.shape)).encode())
    digest.update(np.ascontiguousarray(array).tobytes())

def sample_fingerprint(samples):
    digest = hashlib.sha256()
    for sample in samples:
        digest.update(str(sample.id).encode())
        digest.update(str(sample.split).encode())
        _hash_array(digest, sample.structure.atomic_numbers)
        _hash_array(digest, sample.structure.positions)
        _hash_array(digest, sample.structure.cell)
        _hash_array(digest, sample.structure.pbc)
        for name in ("energy", "forces", "charges", "magmoms"):
            digest.update(name.encode())
            if name in sample.labels:
                _hash_array(digest, sample.labels[name])
            else:
                digest.update(b"<missing>")
    return digest.hexdigest()

dataset_fingerprint = sample_fingerprint(all_samples)
label_frames = {
    name: sum(name in sample.labels for sample in all_samples)
    for name in ("energy", "forces", "charges", "magmoms")
}
print("Source / scanned / accepted / rejected:")
print(dataset.report.source, dataset.report.scanned, dataset.report.accepted, dataset.report.rejected)
print("Split audit       :", dataset.audit.split_counts)
print("Labelled frames   :", label_frames)
print("Dataset SHA-256   :", dataset_fingerprint)
print("Expected SHA-256  :", EXPECTED_DATASET_SHA256)
print("Electronic source :", provenance)

if REQUIRE_REFERENCE_FINGERPRINT and dataset_fingerprint != EXPECTED_DATASET_SHA256:
    raise RuntimeError(
        "MatPES 样本与 JouleWeave 参考训练集不一致，停止不公平比较。"
        f" expected={EXPECTED_DATASET_SHA256}, actual={dataset_fingerprint}. "
        "请检查数据集 revision、seed、过滤条件或使用旧训练的转换缓存。"
    )

Source / scanned / accepted / rejected:
converted-cache 20012 20000 {'isolated_or_empty': 11, 'force_outlier': 1}
Split audit       : {'train': 16299, 'test': 1877, 'valid': 1824}
Labelled frames   : {'energy': 20000, 'forces': 20000, 'charges': 18676, 'magmoms': 18887}
Dataset SHA-256   : e04aa34106cee73aac2d136c5a0a7e8b50f22e079c9968d421b9b05a80d63f13
Expected SHA-256  : e04aa34106cee73aac2d136c5a0a7e8b50f22e079c9968d421b9b05a80d63f13
Electronic source : {'charge_label_scheme': 'ddec6', 'oxidation_label_method': None}


## 4. 对标容量与训练配置

该配置实测约 2.12M 参数，位于旧 compact（约 0.96M）与 foundation（约 6.31M）
之间。三层传播和两个平滑专家提高复杂化学环境的实际容量；`max_ell=3`、相关阶数 3、
完整半径图和所有物理门禁保持不变。

训练协议使用有效 batch 16、完整任务权重、Huber 0.02 和自动参考能拟合。没有为了追求
表面精度关闭保守力、截断物理或对称性。


In [6]:
active_charge_scheme = provenance.get("charge_label_scheme") or "unspecified"
active_oxidation_method = provenance.get("oxidation_label_method")
use_oxidation = active_oxidation_method is not None

model_config = ZynFieldModelConfig.universal(
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_radial=NUM_RADIAL,
    num_attention_heads=NUM_HEADS,
    num_experts=NUM_EXPERTS,
    expert_top_k=NUM_EXPERTS,
    max_ell=3,
    correlation_order=3,
    tensor_product_rank=TENSOR_RANK,
    directional_edge_rank=EDGE_RANK,
    interaction_cutoff_A=CUTOFF_A,
    max_neighbors=None,
    neighbor_chunk_size=1024,
    pair_chemical_rank=TENSOR_RANK,
    edge_kernel_rank=16,
    max_atomic_number=118,
    radial_trainable=True,
    include_pseudotensors=True,
    include_time_odd=False,
    use_spin_vectors=False,
    use_pair_chemical_bias=True,
    use_hybrid_irrep_norm=True,
    use_learned_residual_scales=True,
    use_electronic_depth_context=True,
    use_invariant_scale_context=USE_SCALE_CONTEXT,
    use_invariant_gated_ffn=USE_GATED_FFN,
    use_path_resolved_radial=USE_PATH_RADIAL,
    radial_path_rank=4,
    use_adaptive_rank_gates=USE_ADAPTIVE_RANK,
    use_periodic_table_prior=USE_PERIODIC_PRIOR,
    extensive_state_conditioning=True,
    cutoff_smoothness_order=4,
    use_layer_energy_mixing=True,
    conservative_forces=True,
    use_magmoms=USE_MAGMOMS,
    use_charge_head=USE_CHARGES,
    use_oxidation_states=use_oxidation,
    charge_label_scheme=active_charge_scheme,
    oxidation_label_method=active_oxidation_method,
    use_zbl=True,
    use_dispersion=USE_DISPERSION,
    use_qeq=USE_QEQ,
    qeq_max_atoms=MAX_ATOMS,
    qeq_allow_periodic_minimum_image=False,
    qeq_min_curvature_eV=1.0e-4,
    use_latent_ewald=USE_LATENT_EWALD,
    allow_combined_electrostatics=False,
)
data_config = ZynFieldDataConfig(
    dataset="matpes-pbe-2025.2-external",
    energy_source="labels.energy",
    forces_source="labels.forces",
    stress_source=None,
    magmoms_source="labels.magmoms" if USE_MAGMOMS else None,
    charges_source="labels.charges" if USE_CHARGES else None,
    oxidation_states_source="labels.oxidation_states" if use_oxidation else None,
    total_charge_source="conditions.total_charge",
    charge_label_scheme=active_charge_scheme,
    oxidation_label_method=active_oxidation_method,
    material_types=("crystal",),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    train_ratio=0.8,
    valid_ratio=0.1,
    test_ratio=0.1,
    seed=SEED,
    pin_memory=DEVICE.type == "cuda",
)
run_name = f"zynforge-zenith-jouleweave-parity-{DATA_LIMIT}-{EPOCHS}e-{run_stamp}"
train_config = ZynFieldTrainConfig(
    epochs=EPOCHS,
    learning_rate=3.0e-4,
    min_learning_rate=1.0e-6,
    warmup_epochs=min(5, max(EPOCHS // 5, 1)),
    weight_decay=1.0e-6,
    amsgrad=True,
    energy_weight=1.0,
    force_weight=10.0,
    stress_weight=0.0,
    magmom_weight=0.2 if USE_MAGMOMS else 0.0,
    charge_weight=1.0 if USE_CHARGES else 0.0,
    oxidation_state_weight=0.1 if use_oxidation else 0.0,
    charge_qeq_consistency_weight=0.0,
    strict_electronic_labels=bool(USE_CHARGES or use_oxidation),
    loss="huber",
    huber_delta=0.02,
    gradient_clip_norm=10.0,
    gradient_accumulation=GRAD_ACCUM,
    ema_decay=0.999,
    patience=max(EPOCHS, 10),
    min_delta=1.0e-6,
    reference_fit="auto",
    device=str(DEVICE),
    dtype="float32",
    seed=SEED,
    deterministic=DETERMINISTIC,
    workspace_root=WORKSPACE_ROOT,
    run_name=run_name,
    backbone_learning_rate_scale=1.0,
    embedding_learning_rate_scale=1.0,
    radial_learning_rate_scale=1.0,
    readout_learning_rate_scale=1.0,
    force_bootstrap_fraction=1.0,
    force_ramp_epochs=1,
    electronic_warmup_epochs=0,
    electronic_ramp_epochs=1,
    expert_balance_warmup_epochs=1,
    expert_balance_weight=0.0,
    radial_data_initialization=True,
    radial_init_batches=12,
    radial_init_max_distances=200_000,
    radial_regularization_weight=1.0e-5,
    rank_sparsity_weight=0.0,
    adaptive_loss_balance=False,
    adaptive_gradient_clip=False,
    selection_energy_scale_eV_per_atom=0.10,
    selection_force_scale_eV_per_A=0.20,
    selection_charge_scale_e=0.10,
    selection_magmom_scale_mu_B=0.20,
    taylor_consistency_weight=0.0,
    compile_conservative_path=COMPILE_CONSERVATIVE,
    compile_backend=None,
    compile_dynamic_shapes=True,
    skip_nonfinite_batches=False,
    progress_bar=True,
    progress_update_interval=PROGRESS_INTERVAL,
    progress_leave=True,
    progress_show_memory=True,
)
config = ZynFieldConfig(model=model_config, data=data_config, train=train_config)

architecture_model = ZynFieldPotential(model_config)
efficiency_report = architecture_efficiency_report(architecture_model)
completeness_certificate = formal_completeness_certificate(architecture_model)
assert model_config.architecture_name == "zynforge-zenith"
assert efficiency_report.complete_edge_cg_layers == model_config.num_layers
assert efficiency_report.full_correlation_layers == model_config.num_layers
assert efficiency_report.grace_finite_tree_basis_contained
assert efficiency_report.single_graph_ace_spine
assert efficiency_report.complete_radius_graph
assert efficiency_report.hard_neighbor_cap is None
assert efficiency_report.merged_graded_rms_norm
assert efficiency_report.grade_aware_invariant_edge_routing
assert completeness_certificate.complete_node_product_layers == model_config.num_layers
assert completeness_certificate.finite_truncation_feature_complete
assert not completeness_certificate.separate_additive_energy_path
assert model_config.extensive_state_conditioning
assert not (model_config.use_qeq and model_config.use_latent_ewald)
assert train_config.force_bootstrap_fraction == 1.0
assert train_config.electronic_warmup_epochs == 0
display(pd.Series({**efficiency_report.to_dict(), "profile": "jouleweave-parity"}))
display(pd.Series(completeness_certificate.to_dict()))
del architecture_model
gc.collect()


parameters                                                 1367766
trainable_parameters                                       1367766
irrep_sector_count                                               8
max_ell                                                          3
pseudotensor_sector_count                                        4
time_odd_sector_count                                            0
interaction_layers                                               3
layer_correlation_orders                                 (3, 3, 3)
full_correlation_layers                                          3
directional_edge_layers                                          3
grouped_cg_modules                                               9
grouped_cg_paths                                              1020
grouped_cg_angular_kernels                                     612
shared_geometry                                               True
complete_radius_graph                                         

certificate_name                                     zynforge-zenith-finite-truncation-certificate
architecture_name                                                                  ZynForge Zenith
basis_family                                     analytic-plus-adaptive radial direct sum with ...
coefficient_parameterization                                                            factorized
radial_basis                                     smooth-orthonormal-polynomial-r2dr-direct-sum-...
radial_order                                                                                    16
analytic_radial_orthogonal_under_r2dr                                                         True
analytic_radial_normalized_at_initialization                                                  True
trainable_radial_rescaling                                                                    True
analytic_radial_subspace_retained                                                             True
adaptive_r

0

## 5. 数据模块、优化器更新数与真实反向传播探针

该单元验证每轮优化器更新次数与有效 batch，并执行一次包含保守力二阶反向传播的真实
训练探针。若显存不足，请回到配置单元改成 `batch=1, accumulation=16`，不要增大有效 batch。


In [7]:
workspace = MLWorkspace(WORKSPACE_ROOT)
active_samples = list(all_samples)
datamodule = prepare_zynfield_datamodule(
    data_config,
    workspace=workspace,
    source=active_samples,
)
train_loader = datamodule.train_dataloader()
valid_loader = datamodule.val_dataloader()
split_sizes = {
    "train": len(datamodule.dataset("train")),
    "valid": len(datamodule.dataset("valid")),
    "test": len(datamodule.dataset("test")),
}
assert sum(split_sizes.values()) == len(active_samples)
train_batches = len(train_loader)
valid_batches = len(valid_loader)
optimizer_steps_per_epoch = math.ceil(train_batches / GRAD_ACCUM)
reference_steps_per_epoch = math.ceil(split_sizes["train"] / 16)
assert optimizer_steps_per_epoch == reference_steps_per_epoch
required_seconds_per_batch = 72.0 * 3600.0 / (
    EPOCHS * max(train_batches + valid_batches, 1)
)
print("Split sizes             :", split_sizes)
print("Train/valid batches     :", train_batches, valid_batches)
print("Optimizer steps / epoch :", optimizer_steps_per_epoch)
print("30-epoch updates        :", optimizer_steps_per_epoch * EPOCHS)
print("72 h average target     :", f"{required_seconds_per_batch:.2f} s/batch")
assert train_batches == math.ceil(split_sizes["train"] / BATCH_SIZE)

runtime_probe = {"enabled": RUN_RUNTIME_PROBE}
if RUN_RUNTIME_PROBE:
    probe_batch = next(iter(train_loader))
    probe_batch, probe_inputs = batch_to_model_inputs(probe_batch)
    runtime_model = ZynFieldPotential(model_config).to(DEVICE, dtype=torch.float32).train()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)
        torch.cuda.synchronize(DEVICE)
    started = time.perf_counter()
    try:
        probe_output = runtime_model.energy_and_forces(probe_inputs, create_graph=True)
        probe_loss = probe_output["energy"].square().mean()
        probe_loss = probe_loss + 0.01 * probe_output["forces"].square().mean()
        probe_loss.backward()
        finite_gradients = all(
            parameter.grad is None or bool(torch.isfinite(parameter.grad).all().item())
            for parameter in runtime_model.parameters()
        )
        if not finite_gradients:
            raise FloatingPointError("训练探针产生非有限梯度")
        if DEVICE.type == "cuda":
            torch.cuda.synchronize(DEVICE)
        elapsed = time.perf_counter() - started
        peak_gib = (
            torch.cuda.max_memory_allocated(DEVICE) / 1024**3
            if DEVICE.type == "cuda" else 0.0
        )
        runtime_probe.update({
            "seconds": elapsed,
            "peak_memory_gib": peak_gib,
            "finite_gradients": finite_gradients,
            "within_72h_average": elapsed <= required_seconds_per_batch,
        })
        print("Train-like probe        :", runtime_probe)
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError(
            "训练探针显存不足。请设置 ZYNFORGE_BATCH_SIZE=1、"
            "ZYNFORGE_GRAD_ACCUM=16 后重启内核；不要改变 effective batch=16。"
        ) from exc
    finally:
        del runtime_model, probe_batch, probe_inputs
        if "probe_output" in locals():
            del probe_output
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

preflight = {
    "accepted_samples": len(active_samples),
    "dataset_fingerprint_sha256": dataset_fingerprint,
    "label_frames": label_frames,
    "split_sizes": split_sizes,
    "train_batches": train_batches,
    "valid_batches": valid_batches,
    "micro_batch": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM,
    "effective_batch": BATCH_SIZE * GRAD_ACCUM,
    "optimizer_steps_per_epoch": optimizer_steps_per_epoch,
    "total_optimizer_steps": optimizer_steps_per_epoch * EPOCHS,
    "required_seconds_per_batch_for_72h": required_seconds_per_batch,
    "runtime_probe": runtime_probe,
    "architecture_efficiency": efficiency_report.to_dict(),
}
(RESULTS_ROOT / "training_preflight.json").write_text(
    json.dumps(preflight, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)


Split sizes             : {'train': 16299, 'valid': 1824, 'test': 1877}
Train/valid batches     : 2038 228
Optimizer steps / epoch : 1019
30-epoch updates        : 30570
72 h average target     : 3.81 s/batch
Train-like probe        : {'enabled': True, 'seconds': 2.5220127960201353, 'peak_memory_gib': 0.6607255935668945, 'finite_gradients': True, 'within_72h_average': True}


2164

## 6. 训练、恢复或仅评价

只允许恢复 `zynforge-zenith` 同构 checkpoint。训练器会验证架构字段、保存 RNG、
优化器、调度器、EMA 更新次数和数据加载器状态；最后一个不完整梯度累积窗口按实际
micro-batch 数归一化。


In [ ]:
print({
    "architecture": model_config.architecture_name,
    "profile": "jouleweave-parity",
    "max_ell": model_config.max_ell,
    "product_order": model_config.correlation_order,
    "maximum_body_order_per_layer": model_config.maximum_body_order,
    "layers": model_config.num_layers,
    "parameters": efficiency_report.parameters,
    "effective_batch": BATCH_SIZE * GRAD_ACCUM,
    "updates_per_epoch": optimizer_steps_per_epoch,
    "samples": len(active_samples),
    "split_sizes": split_sizes,
})
started = time.perf_counter()
result = None
if RUN_MODE == "train":
    result = train_zynfield(config, source=active_samples)
    BEST_CHECKPOINT = Path(result.best_checkpoint).resolve()
    LAST_CHECKPOINT = Path(result.last_checkpoint).resolve()
    RUN_DIR = Path(result.run_dir).resolve()
elif RUN_MODE == "resume":
    checkpoint_text = os.environ["ZYNFORGE_RESUME_CHECKPOINT"].strip()
    if not checkpoint_text:
        raise RuntimeError("resume 模式需要 ZYNFORGE_RESUME_CHECKPOINT")
    resume_checkpoint = Path(checkpoint_text).expanduser().resolve()
    result = resume_zynfield(
        config,
        checkpoint=resume_checkpoint,
        source=active_samples,
        total_epochs=EPOCHS,
        in_place=True,
    )
    BEST_CHECKPOINT = Path(result.best_checkpoint).resolve()
    LAST_CHECKPOINT = Path(result.last_checkpoint).resolve()
    RUN_DIR = Path(result.run_dir).resolve()
else:
    checkpoint_text = os.environ["ZYNFORGE_EVALUATE_CHECKPOINT"].strip()
    if not checkpoint_text:
        raise RuntimeError("evaluate 模式需要 ZYNFORGE_EVALUATE_CHECKPOINT")
    BEST_CHECKPOINT = Path(checkpoint_text).expanduser().resolve()
    LAST_CHECKPOINT = BEST_CHECKPOINT
    RUN_DIR = BEST_CHECKPOINT.parent.parent

if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(BEST_CHECKPOINT)
training_seconds = time.perf_counter() - started
HISTORY_FILE = RUN_DIR / "logs" / "history.jsonl"
validate_checkpoint_provenance(
    BEST_CHECKPOINT,
    charge_label_scheme=active_charge_scheme if USE_CHARGES else None,
    oxidation_label_method=active_oxidation_method if use_oxidation else None,
)
print("Elapsed hours / run / best / last:")
print(training_seconds / 3600.0, RUN_DIR, BEST_CHECKPOINT, LAST_CHECKPOINT, sep="\n")


{'architecture': 'zynforge-zenith', 'profile': 'jouleweave-parity', 'max_ell': 3, 'product_order': 3, 'maximum_body_order_per_layer': 4, 'layers': 3, 'parameters': 1367766, 'effective_batch': 16, 'updates_per_epoch': 1019, 'samples': 20000, 'split_sizes': {'train': 16299, 'valid': 1824, 'test': 1877}}


Epoch 001/030 train:   0%|          | 0/2038 [00:00<?, ?batch/s]

## 7. 训练历史与 JouleWeave 目标线

训练历史中的 Zenith force MAE 是全局 component-micro；旧 JouleWeave 日志是
structure-macro，因此这里只把曲线用于观察收敛。真正的对标判断在后续统一评估单元完成。


In [ ]:
history = None
if HISTORY_FILE.is_file():
    rows = [
        json.loads(line)
        for line in HISTORY_FILE.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    history = pd.DataFrame(rows)
elif result is not None:
    history = pd.DataFrame(result.history)

if history is None or history.empty:
    print("没有找到训练历史。")
else:
    display(history.tail(10))
    history.to_csv(RESULTS_ROOT / "training_history.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    for prefix, color in (("train", "#4C78A8"), ("valid", "#E45756")):
        energy_name = f"{prefix}_energy_mae_eV_per_atom"
        force_name = f"{prefix}_force_mae_eV_per_A"
        if energy_name in history:
            axes[0].plot(history["epoch"], history[energy_name], label=prefix, color=color)
        if force_name in history:
            axes[1].plot(history["epoch"], history[force_name], label=prefix, color=color)
    axes[0].axhline(0.1434482418976147, color="black", ls="--", lw=1.2,
                    label="JouleWeave validation target")
    axes[1].axhline(0.2603700757823198, color="black", ls="--", lw=1.2,
                    label="JouleWeave structure-macro target")
    axes[0].set_title("Energy MAE")
    axes[1].set_title("Force MAE (curve metric is component-micro)")
    for axis in axes:
        axis.set_xlabel("Epoch")
        axis.set_ylabel("MAE")
        axis.set_yscale("log")
        axis.grid(alpha=0.25)
        axis.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(RESULTS_ROOT / "training_history.png", dpi=180)
    plt.show()
    if RUN_MODE in {"train", "resume"}:
        assert int(history["epoch"].max()) == EPOCHS, (
            "训练未达到设定 epoch；请从 last checkpoint 恢复。"
        )


## 8. 加载最佳 checkpoint 并检查架构契约

先加载 EMA 权重进行结构检查；后续会在同一验证集上同时评价 EMA 与 raw 权重，按照训练前
固定的无量纲能量/力/电荷/磁矩分数选择最终部署权重，避免因为默认 EMA 选择造成假优势。


In [ ]:
model = load_zynfield(
    BEST_CHECKPOINT,
    device=str(DEVICE),
    dtype="float32",
    use_ema=True,
)
model.eval()
loaded_efficiency = architecture_efficiency_report(model)
loaded_certificate = formal_completeness_certificate(model)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
assert model.config.architecture_name == "zynforge-zenith"
assert loaded_efficiency.single_graph_ace_spine
assert loaded_efficiency.complete_edge_cg_layers == model.config.num_layers
assert loaded_efficiency.full_correlation_layers == model.config.num_layers
assert loaded_efficiency.grace_finite_tree_basis_contained
assert loaded_efficiency.complete_radius_graph
assert loaded_efficiency.hard_neighbor_cap is None
assert loaded_efficiency.merged_graded_rms_norm
assert loaded_efficiency.grade_aware_invariant_edge_routing
assert loaded_certificate.complete_node_product_layers == model.config.num_layers
assert loaded_certificate.finite_truncation_feature_complete
assert loaded_certificate.single_energy_path
assert not loaded_certificate.separate_additive_energy_path
assert parameter_count == loaded_efficiency.parameters
radial_diagnostics = model.radial_basis.diagnostics()
rank_diagnostics = model.effective_rank_statistics()
if RUN_MODE in {"train", "resume"}:
    assert radial_diagnostics["data_initialized"] is True
    assert radial_diagnostics["initialization_count"] > 0
checkpoint_summary = {
    "architecture": model.config.architecture_name,
    "parameters": parameter_count,
    "efficiency": loaded_efficiency.to_dict(),
    "certificate": loaded_certificate.to_dict(),
    "radial": radial_diagnostics,
    "effective_rank": rank_diagnostics,
}
display(pd.Series({
    "architecture": model.config.architecture_name,
    "parameters": parameter_count,
    "max_ell": model.config.max_ell,
    "product_order": model.config.correlation_order,
    "layers": model.config.num_layers,
    "radial_initialized": radial_diagnostics["data_initialized"],
}))


## 9. 统一验证、EMA/raw 选择与独立测试集评价

该单元对每个结构单独累积 force structure-macro，同时保留全部分量的 micro 指标。
JouleWeave 的 `0.260370 eV/Å` 目标只能与这里的 `force_mae_structure_macro` 比较。
最终部署权重只根据验证集选择；测试集在选择完成后评价一次。


In [ ]:
def evaluate_loader(evaluation_model, loader):
    evaluation_model.eval()
    energy_errors = []
    force_errors = []
    force_frame_mae = []
    force_frame_rmse = []
    charge_errors = []
    charge_frame_mae = []
    magmom_errors = []
    magmom_frame_mae = []

    for raw_batch in loader:
        batch, inputs = batch_to_model_inputs(raw_batch)
        with torch.enable_grad():
            output = evaluation_model.energy_and_forces(inputs, create_graph=False)
        natoms = batch["structure"]["natoms"].detach().cpu().numpy().astype(int).reshape(-1)
        e_ref = batch["targets"]["energy"].detach().cpu().numpy().reshape(-1)
        e_pred = output["energy"].detach().cpu().numpy().reshape(-1)
        energy_errors.extend(((e_pred - e_ref) / natoms).tolist())

        f_ref = batch["targets"]["forces"].detach().cpu().numpy()
        f_pred = output["forces"].detach().cpu().numpy()
        f_error = f_pred - f_ref
        force_errors.append(f_error.reshape(-1))

        offset = 0
        for atom_count in natoms:
            frame_error = f_error[offset:offset + atom_count]
            force_frame_mae.append(float(np.mean(np.abs(frame_error))))
            force_frame_rmse.append(float(np.sqrt(np.mean(frame_error**2))))
            offset += atom_count

        charge_output = output.get("partition_charges", output.get("charges"))
        if "charges" in batch["targets"] and charge_output is not None:
            q_ref = batch["targets"]["charges"].detach().cpu().numpy()
            q_pred = charge_output.detach().cpu().numpy()
            q_mask = batch["masks"]["charges"].detach().cpu().numpy().astype(bool)
            q_error = q_pred - q_ref
            charge_errors.extend(q_error[q_mask].tolist())
            offset = 0
            for atom_count in natoms:
                frame_mask = q_mask[offset:offset + atom_count]
                if frame_mask.any():
                    frame_error = q_error[offset:offset + atom_count][frame_mask]
                    charge_frame_mae.append(float(np.mean(np.abs(frame_error))))
                offset += atom_count

        if "magmoms" in batch["targets"] and "magmoms" in output:
            m_ref = batch["targets"]["magmoms"].detach().cpu().numpy()
            m_pred = output["magmoms"].detach().cpu().numpy()
            m_mask = batch["masks"]["magmoms"].detach().cpu().numpy().astype(bool)
            m_error = m_pred - m_ref
            magmom_errors.extend(m_error[m_mask].tolist())
            offset = 0
            for atom_count in natoms:
                frame_mask = m_mask[offset:offset + atom_count]
                if frame_mask.any():
                    frame_error = m_error[offset:offset + atom_count][frame_mask]
                    magmom_frame_mae.append(float(np.mean(np.abs(frame_error))))
                offset += atom_count

    energy_errors = np.asarray(energy_errors)
    force_errors = np.concatenate(force_errors)
    metrics = {
        "frames": int(len(energy_errors)),
        "energy_mae_eV_per_atom": float(np.mean(np.abs(energy_errors))),
        "energy_rmse_eV_per_atom": float(np.sqrt(np.mean(energy_errors**2))),
        "force_mae_component_micro_eV_per_A": float(np.mean(np.abs(force_errors))),
        "force_rmse_component_micro_eV_per_A": float(np.sqrt(np.mean(force_errors**2))),
        "force_mae_structure_macro_eV_per_A": float(np.mean(force_frame_mae)),
        "force_rmse_structure_macro_eV_per_A": float(np.mean(force_frame_rmse)),
    }
    if charge_errors:
        charge_errors = np.asarray(charge_errors)
        metrics.update({
            "charge_mae_atom_micro_e": float(np.mean(np.abs(charge_errors))),
            "charge_rmse_atom_micro_e": float(np.sqrt(np.mean(charge_errors**2))),
            "charge_mae_labelled_frame_macro_e": float(np.mean(charge_frame_mae)),
        })
    if magmom_errors:
        magmom_errors = np.asarray(magmom_errors)
        metrics.update({
            "magmom_mae_atom_micro_mu_B": float(np.mean(np.abs(magmom_errors))),
            "magmom_rmse_atom_micro_mu_B": float(np.sqrt(np.mean(magmom_errors**2))),
            "magmom_mae_labelled_frame_macro_mu_B": float(np.mean(magmom_frame_mae)),
        })
    return metrics

def fixed_validation_score(metrics):
    score = metrics["energy_mae_eV_per_atom"] / 0.10
    score += metrics["force_mae_component_micro_eV_per_A"] / 0.20
    if USE_CHARGES:
        score += metrics["charge_mae_atom_micro_e"] / 0.10
    if USE_MAGMOMS:
        score += metrics["magmom_mae_atom_micro_mu_B"] / 0.20
    return float(score)

validation_ema = evaluate_loader(model, datamodule.val_dataloader())
raw_model = load_zynfield(
    BEST_CHECKPOINT,
    device=str(DEVICE),
    dtype="float32",
    use_ema=False,
).eval()
validation_raw = evaluate_loader(raw_model, datamodule.val_dataloader())
ema_score = fixed_validation_score(validation_ema)
raw_score = fixed_validation_score(validation_raw)

if raw_score < ema_score:
    selected_weight_source = "raw"
    model = raw_model
    validation_metrics = validation_raw
else:
    selected_weight_source = "ema"
    validation_metrics = validation_ema
    del raw_model
    gc.collect()

test_metrics = {"status": "skipped"}
if RUN_FULL_TEST:
    test_metrics = evaluate_loader(model, datamodule.test_dataloader())

JOULEWEAVE_TARGETS = {
    "energy_mae_eV_per_atom": 0.1434482418976147,
    "force_mae_structure_macro_eV_per_A": 0.2603700757823198,
    "charge_mae_labelled_frame_macro_e": 0.0700971288339923,
    "magmom_mae_labelled_frame_macro_mu_B": 0.16759990326072305,
}
parity_report = {
    "selected_weight_source": selected_weight_source,
    "ema_validation_score": ema_score,
    "raw_validation_score": raw_score,
    "targets": JOULEWEAVE_TARGETS,
    "validation": validation_metrics,
    "energy_pass": (
        validation_metrics["energy_mae_eV_per_atom"]
        <= JOULEWEAVE_TARGETS["energy_mae_eV_per_atom"]
    ),
    "force_pass": (
        validation_metrics["force_mae_structure_macro_eV_per_A"]
        <= JOULEWEAVE_TARGETS["force_mae_structure_macro_eV_per_A"]
    ),
}
parity_report["joint_energy_force_pass"] = bool(
    parity_report["energy_pass"] and parity_report["force_pass"]
)

print("EMA validation score:", ema_score)
print("Raw validation score:", raw_score)
print("Selected weights     :", selected_weight_source)
display(pd.DataFrame({
    "Zenith validation": validation_metrics,
    "JouleWeave target": {
        **JOULEWEAVE_TARGETS,
        "frames": np.nan,
    },
}).T)
print("Joint energy/force parity:", parity_report["joint_energy_force_pass"])
if RUN_FULL_TEST:
    display(pd.Series(test_metrics, name="independent_test"))

(RESULTS_ROOT / "validation_metrics.json").write_text(
    json.dumps(validation_metrics, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
(RESULTS_ROOT / "test_metrics.json").write_text(
    json.dumps(test_metrics, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
(RESULTS_ROOT / "jouleweave_parity.json").write_text(
    json.dumps(parity_report, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
if REQUIRE_PARITY and not parity_report["joint_energy_force_pass"]:
    raise AssertionError(
        "30-epoch checkpoint 未同时达到 JouleWeave 能量与 structure-macro 力目标；"
        "详细差距已写入 jouleweave_parity.json。"
    )


## 10. 最终部署权重的双精度物理验证

这里验证刚才依据验证集选出的 EMA 或 raw 权重，而不是只验证未训练模型。二阶检查仅在
小于等于 12 个原子的测试结构上执行，以控制 Notebook 时间。


In [ ]:
physics_checks = {"status": "skipped"}
if RUN_PHYSICS_CHECK:
    diagnostic_model = load_zynfield(
        BEST_CHECKPOINT,
        device="cpu",
        dtype="float64",
        use_ema=(selected_weight_source == "ema"),
    ).eval()
    diagnostic_sample = datamodule.dataset("test")[0]
    diagnostic_batch = jouleweave_collate([diagnostic_sample])

    def to_float64_cpu(value):
        if torch.is_tensor(value):
            value = value.detach().cpu()
            return value.to(torch.float64) if value.is_floating_point() else value
        if isinstance(value, dict):
            return {key: to_float64_cpu(item) for key, item in value.items()}
        return value

    diagnostic_batch = to_float64_cpu(diagnostic_batch)
    diagnostic_inputs = {
        **diagnostic_batch["structure"],
        **diagnostic_batch["conditions"],
    }
    force_report = check_conservative_forces(diagnostic_model, diagnostic_inputs)
    permutation_report = check_permutation_translation_invariance(
        diagnostic_model, diagnostic_inputs
    )
    generator = torch.Generator().manual_seed(SEED + 1729)
    random_matrix, _ = torch.linalg.qr(
        torch.randn((3, 3), generator=generator, dtype=torch.float64)
    )
    if torch.det(random_matrix) < 0:
        random_matrix[:, 0] *= -1.0
    rotation_report = check_o3_equivariance(
        diagnostic_model, diagnostic_inputs, random_matrix
    )
    reflection = torch.diag(torch.tensor([-1.0, 1.0, 1.0], dtype=torch.float64))
    reflection_report = check_o3_equivariance(
        diagnostic_model, diagnostic_inputs, reflection @ random_matrix
    )
    for report in (force_report, permutation_report, rotation_report, reflection_report):
        assert report.passed, report
    physics_checks = {
        "force": asdict(force_report),
        "permutation_translation": asdict(permutation_report),
        "rotation": asdict(rotation_report),
        "reflection": asdict(reflection_report),
    }
    atom_count = int(diagnostic_inputs["pos"].shape[0])
    if RUN_SECOND_ORDER_CHECK and atom_count <= 12:
        hessian_report = check_hessian_symmetry(diagnostic_model, diagnostic_inputs)
        assert hessian_report.passed, hessian_report
        physics_checks["hessian"] = asdict(hessian_report)
        pbc = diagnostic_inputs.get("pbc")
        if pbc is not None and bool(torch.all(pbc).item()):
            stress_report = check_stress_energy_derivative(
                diagnostic_model, diagnostic_inputs
            )
            assert stress_report.passed, stress_report
            physics_checks["stress"] = asdict(stress_report)
    elif RUN_SECOND_ORDER_CHECK:
        physics_checks["second_order"] = {
            "status": "skipped",
            "reason": f"{atom_count} atoms exceeds the safe notebook threshold of 12",
        }
    display(pd.json_normalize(physics_checks).T)
    (RESULTS_ROOT / "physics_checks.json").write_text(
        json.dumps(physics_checks, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    del diagnostic_model
    gc.collect()
else:
    print("双精度 checkpoint 物理验证已关闭。")


## 11. 可选 ASE 接口与材料级验证边界

通过架构和回归门禁不代表具体材料已经具有可靠的声子、势垒、相排序或长时间 MD。
投产前仍需对目标体系完成 NVE 漂移、EOS、声子、NEB、相稳定性和真实 LAMMPS
长轨迹验证。下面只检查 ASE 能量/力/电子属性接口。


In [ ]:
if RUN_ASE_CHECK:
    try:
        from ase import Atoms
    except ImportError:
        Atoms = None
    if Atoms is None:
        print("ASE 未安装，跳过接口验证。")
    else:
        sample = datamodule.dataset("test")[0]
        structure = sample.structure
        atoms = Atoms(
            numbers=np.asarray(structure.atomic_numbers),
            positions=np.asarray(structure.positions),
            cell=np.asarray(structure.cell),
            pbc=np.asarray(structure.pbc),
        )
        atoms.info["total_charge"] = float(sample.conditions.get("total_charge", 0.0))
        atoms.calc = zynfield_calculator(
            model,
            device=str(DEVICE),
            dtype="float32",
            analytic_stress=False,
            compile_model=False,
        )
        print("Energy (eV):", atoms.get_potential_energy())
        print("Max |force| (eV/Å):", float(np.linalg.norm(atoms.get_forces(), axis=1).max()))
        for property_name in ("charges", "magmoms", "oxidation_states"):
            try:
                print(property_name, np.asarray(atoms.calc.get_property(property_name, atoms)))
            except Exception as exc:
                print(property_name, "unavailable:", type(exc).__name__, exc)
else:
    print("ASE 接口验证已关闭。")


## 12. 保存完整训练、对标与物理验收摘要


In [ ]:
final_summary = {
    "architecture": "zynforge-zenith",
    "profile": "jouleweave-parity",
    "run_mode": RUN_MODE,
    "epochs": EPOCHS,
    "elapsed_hours_in_this_session": training_seconds / 3600.0,
    "project_root": str(PROJECT_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "run_dir": str(RUN_DIR),
    "best_checkpoint": str(BEST_CHECKPOINT),
    "last_checkpoint": str(LAST_CHECKPOINT),
    "history_file": str(HISTORY_FILE),
    "selected_weight_source": selected_weight_source,
    "device": str(DEVICE),
    "accelerator": gpu_name,
    "data_limit": DATA_LIMIT,
    "dataset_fingerprint_sha256": dataset_fingerprint,
    "expected_dataset_fingerprint_sha256": EXPECTED_DATASET_SHA256,
    "scanned_samples": int(dataset.report.scanned),
    "accepted_samples": len(active_samples),
    "label_frames": label_frames,
    "split_sizes": split_sizes,
    "preflight": preflight,
    "source_physics": source_physics,
    "source_regression": source_regression,
    "checkpoint": checkpoint_summary,
    "model_config": asdict(model_config),
    "train_config": asdict(train_config),
    "ema_validation": validation_ema,
    "raw_validation": validation_raw,
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
    "jouleweave_parity": parity_report,
    "physics_checks": physics_checks,
    "selection_policy": {
        "dataset_identity": "strict-sha256-by-default",
        "effective_batch": 16,
        "weight_variant": "fixed-validation-score",
        "periodic_qeq": "rejected",
        "combined_qeq_latent_ewald": "rejected",
        "optional_refinements": "evidence-gated-off",
    },
}
summary_path = RESULTS_ROOT / "training_summary.json"
summary_path.write_text(
    json.dumps(final_summary, indent=2, ensure_ascii=False, default=str) + "\n",
    encoding="utf-8",
)
print("Summary :", summary_path)
print("Parity  :", RESULTS_ROOT / "jouleweave_parity.json")
print("Passed  :", parity_report["joint_energy_force_pass"])
print("Weights :", selected_weight_source)
print("Best    :", BEST_CHECKPOINT)
print("Last    :", LAST_CHECKPOINT)
